In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
import os
sys.path.append(os.path.abspath(".."))

import joblib
import pandas as pd

from src.data import load_matches
from src.elo import prepare_matches
from src.simulation import simulate_match, simulate_group, simulate_group_stage, simulate_knockout_match, construct_round32, simulate_knockout_round, construct_round16, construct_QF, construct_SF, simulate_tournament
from src.features import build_features, assign_third_place_slots

#Load everything from earlier notebooks
country_elo = joblib.load('../models/final_elo.joblib')
model_h = joblib.load('../models/home_goals_model.joblib')
model_a = joblib.load('../models/away_goals_model.joblib')
features = joblib.load('../models/model_features.joblib')

#Make a dataframe containing all the games in the group stage
group_stage_matches = load_matches()
group_stage_matches = prepare_matches(group_stage_matches, "2026-06-10", "2026-06-28")
df_confederations = pd.read_csv('../data/FIFA_confederations.csv')

wc_groups = {
    "A": ["Mexico", "South Korea", "South Africa", "Czech Republic"],
    "B": ["Canada", "Switzerland", "Qatar", "Bosnia and Herzegovina"],
    "C": ["Brazil", "Morocco", "Haiti", "Scotland"],
    "D": ["United States", "Paraguay", "Australia", "Turkey"],
    "E": ["Germany", "Curaçao", "Ivory Coast", "Ecuador"],
    "F": ["Netherlands", "Japan", "Sweden", "Tunisia"],
    "G": ["Belgium", "Egypt", "Iran", "New Zealand"],
    "H": ["Spain", "Cape Verde", "Saudi Arabia", "Uruguay"],
    "I": ["France", "Senegal", "Iraq", "Norway"],
    "J": ["Argentina", "Algeria", "Austria", "Jordan"],
    "K": ["Portugal", "DR Congo", "Uzbekistan", "Colombia"],
    "L": ["England", "Croatia", "Ghana", "Panama"],
}

In [75]:
team_to_group = {team: group for group, teams in wc_groups.items() for team in teams}
team_to_confederation = dict(zip(df_confederations["nation"], df_confederations["confederation"]))

# Create group_stages equivalent
rows = []
for group, teams in wc_groups.items():
    for i, team in enumerate(teams, 1):
        rows.append({"group": group, "position": i, "nation": team})
df_groups = pd.DataFrame(rows)

df_group_fixtures = group_stage_matches.copy()
df_group_fixtures["group"] = df_group_fixtures["home_team"].map(team_to_group)

#Example run
x=build_features('Argentina', 'Brazil', country_elo, team_to_confederation, features)
x

,const,home_elo,away_elo,neutral,tournament_weight,home_confederation_AFC,home_confederation_CAF,home_confederation_CONCACAF,home_confederation_CONMEBOL,home_confederation_OFC,home_confederation_UEFA,home_confederation_Unknown,away_confederation_AFC,away_confederation_CAF,away_confederation_CONCACAF,away_confederation_CONMEBOL,away_confederation_OFC,away_confederation_UEFA,away_confederation_Unknown
0,1.0,2114.688178,1992.738734,1.0,5.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0


In [76]:
#One match dummy test
y = simulate_match('Argentina', 'Brazil', model_h, model_a, country_elo, team_to_confederation, features)
y

{'home_team': 'Argentina',
 'away_team': 'Brazil',
 'home_goals': 0,
 'away_goals': 3,
 'result': 'away_win',
 'winner': 'Brazil'}

In [77]:
#One group dummy test, with matches displayed
group_a_table, group_a_matches = simulate_group("C", df_groups, df_group_fixtures, model_h, model_a, country_elo, team_to_confederation, features)

display(group_a_matches)
display(group_a_table)

,home_team,away_team,home_goals,away_goals,result,winner
0,Brazil,Morocco,0,3,away_win,Morocco
1,Haiti,Scotland,0,0,draw,NaN
2,Scotland,Morocco,0,0,draw,NaN
3,Brazil,Haiti,2,0,home_win,Brazil
4,Scotland,Brazil,0,2,away_win,Brazil
5,Morocco,Haiti,2,0,home_win,Morocco


,team,played,wins,draws,losses,goals_for,goals_against,goal_difference,points,group_rank,group
0,Morocco,3,2,1,0,5,0,5,7,1,C
1,Brazil,3,2,0,1,4,3,1,6,2,C
2,Scotland,3,0,2,1,0,2,-2,2,3,C
3,Haiti,3,0,1,2,0,4,-4,1,4,C


In [78]:
#Full group stage sim
group_stage = simulate_group_stage(df_groups, df_group_fixtures, model_h, model_a, country_elo, team_to_confederation, features)
group_stage_result = group_stage[0]

top2 = group_stage_result.groupby('group').head(2).copy() #Teams that placed top 2 in their group which qualifies
third = group_stage_result.groupby('group').nth(2).copy() #All teams that placed third (only 8 of them move on)
best8_third = third.sort_values(
    ['points', 'goal_difference', 'goals_for'], 
    ascending=[False, False, False]
).head(8).reset_index(drop=True)

best8_third['third_place_rank'] = best8_third.index + 1

display(group_stage_result)

,team,played,wins,draws,losses,goals_for,goals_against,goal_difference,points,group_rank,group
0,Mexico,3,3,0,0,6,0,6,9,1,A
1,South Korea,3,2,0,1,4,3,1,6,2,A
2,South Africa,3,1,0,2,3,4,-1,3,3,A
3,Czech Republic,3,0,0,3,0,6,-6,0,4,A
4,Switzerland,3,1,2,0,3,1,2,5,1,B
5,Canada,3,1,2,0,4,3,1,5,2,B
6,Qatar,3,1,0,2,3,5,-2,3,3,B
7,Bosnia and Herzegovina,3,0,2,1,4,5,-1,2,4,B
8,Morocco,3,2,1,0,2,0,2,7,1,C
9,Brazil,3,2,0,1,12,3,9,6,2,C


In [79]:
round_of_32 = pd.concat([top2, best8_third])
print("All the teams that made it into the round of 32")
round_of_32

All the teams that made it into the round of 32


,team,played,wins,draws,losses,goals_for,goals_against,goal_difference,points,group_rank,group,third_place_rank
0,Mexico,3,3,0,0,6,0,6,9,1,A,NaN
1,South Korea,3,2,0,1,4,3,1,6,2,A,NaN
4,Switzerland,3,1,2,0,3,1,2,5,1,B,NaN
5,Canada,3,1,2,0,4,3,1,5,2,B,NaN
8,Morocco,3,2,1,0,2,0,2,7,1,C,NaN
9,Brazil,3,2,0,1,12,3,9,6,2,C,NaN
12,United States,3,2,0,1,4,1,3,6,1,D,NaN
13,Turkey,3,1,2,0,4,3,1,5,2,D,NaN
16,Ecuador,3,2,1,0,5,1,4,7,1,E,NaN
17,Ivory Coast,3,1,1,1,4,4,0,4,2,E,NaN


In [80]:
#One knockout round dummy test
rd32_teams = {}
for team in top2.itertuples(index=False):
        rd32_teams[f'{team.group_rank}{team.group}'] = team.team
        
thirds = assign_third_place_slots(best8_third)
rd32_teams |= thirds

x = construct_round32(rd32_teams)
simulate_knockout_match(x[73][0], x[73][1], model_h, model_a, country_elo, team_to_confederation, features)

{'home_team': 'South Korea',
 'away_team': 'Canada',
 'home_goals': 0,
 'away_goals': 1,
 'result': 'away_win',
 'result_type': 'regular_time',
 'winner': 'Canada',
 'loser': 'South Korea',
 'home_win_prob': np.float64(0.3406008127431053),
 'draw_prob': np.float64(0.2765824567309899),
 'away_win_prob': np.float64(0.3828068064965937)}

In [81]:
#Round of 32, with results and probabilities of results
winners, results = simulate_knockout_round(x, model_h, model_a, country_elo, team_to_confederation, features)

display(results)

,home_team,away_team,home_goals,away_goals,result_type,winner,loser,home_win_prob,draw_prob,away_win_prob
0,South Korea,Canada,1,2,regular_time,Canada,South Korea,0.340601,0.276582,0.382807
1,Ecuador,Paraguay,1,0,regular_time,Ecuador,Paraguay,0.481782,0.278164,0.240041
2,Japan,Brazil,4,2,regular_time,Japan,Brazil,0.291794,0.288932,0.419268
3,Morocco,Netherlands,0,1,regular_time,Netherlands,Morocco,0.330371,0.307950,0.361676
4,Senegal,Spain,1,3,regular_time,Spain,Senegal,0.111484,0.219313,0.669102
5,Ivory Coast,France,2,1,regular_time,Ivory Coast,France,0.106707,0.210108,0.683049
6,Mexico,Norway,1,1,OT/Pens,Norway,Mexico,0.427296,0.285105,0.287591
7,Panama,Colombia,1,1,OT/Pens,Colombia,Panama,0.188726,0.247898,0.563327
8,United States,Algeria,0,0,OT/Pens,Algeria,United States,0.435514,0.291464,0.273016
9,Iran,South Africa,2,0,regular_time,Iran,South Africa,0.687609,0.206842,0.105397


In [82]:
#Round of 16
r16_matchups = construct_round16(winners)

winners, results = simulate_knockout_round(r16_matchups, model_h, model_a, country_elo, team_to_confederation, features)

display(results)

,home_team,away_team,home_goals,away_goals,result_type,winner,loser,home_win_prob,draw_prob,away_win_prob
0,Ecuador,Spain,0,0,OT/Pens,Spain,Ecuador,0.200692,0.270840,0.528449
1,Canada,Japan,1,2,regular_time,Japan,Canada,0.284711,0.293177,0.422107
2,Netherlands,Ivory Coast,0,0,OT/Pens,Netherlands,Ivory Coast,0.558617,0.262470,0.178886
3,Norway,Colombia,1,4,regular_time,Colombia,Norway,0.208121,0.259591,0.532257
4,England,Jordan,2,1,regular_time,England,Jordan,0.602921,0.244511,0.152517
5,Algeria,Iran,0,1,regular_time,Iran,Algeria,0.323663,0.312883,0.363452
6,Argentina,Belgium,2,0,regular_time,Argentina,Belgium,0.639367,0.230572,0.129985
7,New Zealand,Uzbekistan,2,0,regular_time,New Zealand,Uzbekistan,0.306970,0.237703,0.455247


In [ ]:
#quarter Finals
QF_matchsups = construct_QF(winners)

winners, results = simulate_knockout_round(QF_matchsups, model_h, model_a, country_elo, team_to_confederation, features)

display(results)

,home_team,away_team,home_goals,away_goals,result_type,winner,loser,home_win_prob,draw_prob,away_win_prob
0,Spain,Japan,0,0,OT/Pens,Spain,Japan,0.483625,0.289198,0.227168
1,England,Iran,0,0,OT/Pens,England,Iran,0.500661,0.279729,0.219597
2,Netherlands,Colombia,1,3,regular_time,Colombia,Netherlands,0.231286,0.268993,0.499700
3,Argentina,New Zealand,1,0,regular_time,Argentina,New Zealand,0.901144,0.075170,0.019225


In [84]:
#Semi Finals

SF_matchsups = construct_SF(winners)

winners, results = simulate_knockout_round(SF_matchsups, model_h, model_a, country_elo, team_to_confederation, features)

display(results)

,home_team,away_team,home_goals,away_goals,result_type,winner,loser,home_win_prob,draw_prob,away_win_prob
0,Spain,England,1,1,OT/Pens,Spain,England,0.428756,0.291629,0.279610
1,Colombia,Argentina,0,0,OT/Pens,Argentina,Colombia,0.233532,0.281325,0.485131


In [85]:
#Finals
winner, results = simulate_knockout_round({103: (winners[101], winners[102])}, model_h, model_a, country_elo, team_to_confederation, features)

display(results)

print(f'{winner[103]} won the World Cup')

,home_team,away_team,home_goals,away_goals,result_type,winner,loser,home_win_prob,draw_prob,away_win_prob
0,Spain,Argentina,0,0,OT/Pens,Spain,Argentina,0.290618,0.291795,0.417581


Spain won the World Cup


In [ ]:
#Simulate whole tournament at once
r = simulate_tournament(model_h, model_a, country_elo, team_to_confederation, features, df_groups, df_group_fixtures)
summary = r['summary']
print(f"Winner: {summary['winner']}")
print(f"loser: {summary['runner_up']}")
print(f"sf_teams: {summary['sf_teams']}")
print(f"qf_teams: {summary['qf_teams']}")

display(r['group_tables'])
display(r['group_matches'])
display(r['bracket'])


Winner: Spain
loser: Argentina
sf_teams: ['Ecuador', 'Spain', 'Croatia', 'Argentina']
qf_teams: ['Ecuador', 'Japan', 'Ivory Coast', 'Croatia', 'Spain', 'Turkey', 'Argentina', 'Colombia']


,team,played,wins,draws,losses,goals_for,goals_against,goal_difference,points,group_rank,group
0,South Korea,3,3,0,0,8,3,5,9,1,A
1,Mexico,3,2,0,1,14,2,12,6,2,A
2,South Africa,3,1,0,2,7,10,-3,3,3,A
3,Czech Republic,3,0,0,3,4,18,-14,0,4,A
4,Canada,3,3,0,0,5,1,4,9,1,B
5,Switzerland,3,2,0,1,3,1,2,6,2,B
6,Bosnia and Herzegovina,3,1,0,2,2,4,-2,3,3,B
7,Qatar,3,0,0,3,1,5,-4,0,4,B
8,Brazil,3,3,0,0,13,2,11,9,1,C
9,Morocco,3,1,1,1,3,3,0,4,2,C


,home_team,away_team,home_goals,away_goals,result,winner
0,Mexico,South Africa,4,0,home_win,Mexico
1,South Korea,Czech Republic,3,1,home_win,South Korea
2,Czech Republic,South Africa,2,5,away_win,South Africa
3,Mexico,South Korea,0,1,away_win,South Korea
4,Mexico,Czech Republic,10,1,home_win,Mexico
...,...,...,...,...,...,...
67,Ghana,Panama,1,1,draw,NaN
68,England,Ghana,8,1,home_win,England
69,Panama,Croatia,0,0,draw,NaN
70,Panama,England,0,1,away_win,England


,home_team,away_team,home_goals,away_goals,result_type,winner,loser,home_win_prob,draw_prob,away_win_prob,round
0,Mexico,Switzerland,1,1,OT/Pens,Switzerland,Mexico,0.456492,0.280172,0.263325,R32
1,Ecuador,Sweden,2,2,OT/Pens,Ecuador,Sweden,0.664452,0.214793,0.120625,R32
2,Japan,Morocco,1,0,regular_time,Japan,Morocco,0.434642,0.302096,0.263258,R32
3,Brazil,Netherlands,1,2,regular_time,Netherlands,Brazil,0.455730,0.288553,0.255709,R32
4,Senegal,United States,0,0,OT/Pens,Senegal,United States,0.429970,0.285051,0.284971,R32
5,Ivory Coast,Iraq,2,0,regular_time,Ivory Coast,Iraq,0.427744,0.302373,0.269880,R32
6,South Korea,Germany,0,2,regular_time,Germany,South Korea,0.253521,0.278213,0.468253,R32
7,Croatia,Uzbekistan,2,0,regular_time,Croatia,Uzbekistan,0.446109,0.288299,0.265584,R32
8,Turkey,Bosnia and Herzegovina,2,1,regular_time,Turkey,Bosnia and Herzegovina,0.666151,0.208653,0.125022,R32
9,Belgium,Cape Verde,0,0,OT/Pens,Belgium,Cape Verde,0.676694,0.210942,0.112225,R32


In [ ]:
joblib.dump(df_groups, "../models/df_groups.joblib")
joblib.dump(df_group_fixtures, "d../models/f_groups_fixtures.joblib")
joblib.dump(team_to_confederation, "../models/team_to_confed.joblib")